# Temperature Scaling for DistilBERT

DistilBERT places >95% of its predictions within 0.01 of either 0 or 1 — the score distribution is extremely bimodal. This is typical for fine-tuned transformer classifiers and is the standard motivation for **temperature scaling** (Guo et al., 2017, *On Calibration of Modern Neural Networks*).

Temperature scaling fits a single scalar `T` on a held-out calibration set so that `softmax(logits / T)` produces well-calibrated probabilities. `T > 1` softens overconfident predictions; `T < 1` sharpens underconfident ones.

This notebook:

1. Carves 15% of training data as a held-out calibration set
2. Retrains DistilBERT on the remaining 85%
3. Computes Expected Calibration Error (ECE) before scaling
4. Fits `T` via L-BFGS optimization of negative log-likelihood
5. Recomputes ECE after scaling
6. Re-plots calibration diagram, score distribution, and ROC

Threshold-free metrics (AP, ROC-AUC) are preserved exactly because temperature scaling is monotonic.

**Requires GPU runtime.** ~10 minutes total.

## Environment setup

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas pyarrow transformers accelerate matplotlib

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=True)
DATA_DIR = Path("/content/drive/MyDrive/biology_refusal/data")
OUT_DIR  = Path("/content/drive/MyDrive/biology_refusal/models")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR:  {OUT_DIR}")

In [ ]:
# Requires GPU runtime. Same train/test parquets as previous notebooks.

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, average_precision_score, roc_auc_score,
    roc_curve, f1_score, precision_recall_curve,
)
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 160,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "font.size": 10,
})

FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

train_df = pd.read_parquet(DATA_DIR / "train.parquet")
test_df  = pd.read_parquet(DATA_DIR / "test.parquet")
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

## Replicate the test-set cleanup

In [ ]:
# Need to mirror exactly what the cleanup notebook did: drop test examples
# with max train-similarity >= 0.90 so we're evaluating on the same clean
# test set the comparison results were reported on.

print("\nReplicating cleanup: drop near-duplicate test examples (sim >= 0.90)...")
encoder = SentenceTransformer("all-MiniLM-L6-v2")
train_emb = encoder.encode(train_df["text"].tolist(), batch_size=64,
                            show_progress_bar=False, convert_to_numpy=True)
test_emb  = encoder.encode(test_df["text"].tolist(), batch_size=64,
                            show_progress_bar=False, convert_to_numpy=True)
sim_matrix = cosine_similarity(test_emb, train_emb)
max_sim_per_test = sim_matrix.max(axis=1)
near_dup_mask = max_sim_per_test >= 0.90
test_df_clean = test_df.loc[~near_dup_mask].reset_index(drop=True)
print(f"  Test after cleanup: {len(test_df_clean)} (was {len(test_df)})")

## Carve out a held-out calibration split from training data

In [ ]:
# Standard practice: temperature is fit on data NOT seen during training, so
# the calibration estimate isn't optimistically biased by overfit. We take
# 15% of training data, stratified by label so the calibration set has the
# same class balance.

from sklearn.model_selection import train_test_split

train_idx, calib_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=0.15,
    random_state=SEED,
    stratify=train_df["label"].to_numpy(),
)
train_fit_df  = train_df.iloc[train_idx].reset_index(drop=True)
calib_df      = train_df.iloc[calib_idx].reset_index(drop=True)
print(f"\nTrain (for fitting):    {len(train_fit_df)}  "
      f"(pos={train_fit_df['label'].sum()}, neg={(train_fit_df['label']==0).sum()})")
print(f"Calibration (held-out): {len(calib_df)}  "
      f"(pos={calib_df['label'].sum()}, neg={(calib_df['label']==0).sum()})")

## Retrain DistilBERT on the reduced training set

In [ ]:
# We must retrain because the original DistilBERT saw the calibration data
# during fit. Same hyperparameters as the modeling notebook so this is an
# apples-to-apples comparison with the original DistilBERT result.

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx],
        }

y_train_fit = train_fit_df["label"].to_numpy()
y_calib     = calib_df["label"].to_numpy()
y_test      = test_df_clean["label"].to_numpy()

train_fit_ds = TextDataset(train_fit_df["text"].tolist(), y_train_fit, tokenizer)
calib_ds     = TextDataset(calib_df["text"].tolist(),     y_calib,     tokenizer)
test_ds      = TextDataset(test_df_clean["text"].tolist(), y_test,     tokenizer)

n_neg = (y_train_fit == 0).sum()
n_pos = (y_train_fit == 1).sum()
class_weights = torch.tensor(
    [len(y_train_fit) / (2 * n_neg), len(y_train_fit) / (2 * n_pos)],
    dtype=torch.float,
).to(device)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=class_weights)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=str(OUT_DIR / "distilbert_tempscale"),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
    logging_steps=50, save_strategy="no", eval_strategy="no",
    seed=SEED, report_to="none", fp16=(device == "cuda"),
)

trainer = WeightedTrainer(model=model, args=training_args, train_dataset=train_fit_ds)
print("\nTraining DistilBERT on reduced training set (calibration set held out)...")
trainer.train()

## Extract logits for calib and test sets

In [ ]:
# Temperature scaling operates on logits, not probabilities. We need raw
# pre-softmax outputs.

def get_logits(model, dataset, batch_size=32):
    model.eval()
    all_logits = []
    with torch.no_grad():
        for i in range(0, len(dataset), batch_size):
            batch = [dataset[j] for j in range(i, min(i + batch_size, len(dataset)))]
            input_ids      = torch.stack([b["input_ids"] for b in batch]).to(device)
            attention_mask = torch.stack([b["attention_mask"] for b in batch]).to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            all_logits.append(logits.cpu().numpy())
    return np.concatenate(all_logits, axis=0)

print("\nExtracting logits...")
calib_logits = get_logits(model, calib_ds)
test_logits  = get_logits(model, test_ds)
print(f"  Calib logits shape: {calib_logits.shape}")
print(f"  Test logits shape:  {test_logits.shape}")

## ECE (Expected Calibration Error) helper

In [ ]:
# ECE = sum over bins of |bin_accuracy - bin_confidence| * bin_weight
# Lower is better. Standard 15-bin formulation from Guo et al. 2017.

def expected_calibration_error(probs_pos_class, labels, n_bins=15):
    """probs_pos_class: predicted P(class=1). labels: 0/1 true labels."""
    confidences = np.where(labels == 1, probs_pos_class, 1 - probs_pos_class)
    predictions = (probs_pos_class >= 0.5).astype(int)
    accuracies  = (predictions == labels).astype(float)

    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_details = []
    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i+1]
        if i == n_bins - 1:
            in_bin = (confidences >= lo) & (confidences <= hi)
        else:
            in_bin = (confidences >= lo) & (confidences < hi)
        n_in = in_bin.sum()
        if n_in > 0:
            bin_acc  = accuracies[in_bin].mean()
            bin_conf = confidences[in_bin].mean()
            ece += (n_in / len(labels)) * abs(bin_acc - bin_conf)
            bin_details.append((lo, hi, n_in, bin_acc, bin_conf))
    return ece, bin_details

## Pre-scaling diagnostics

In [ ]:
# Compute softmax probabilities at T=1 (no scaling) on calibration and test
# sets. This is the baseline DistilBERT behavior — the "extreme confidence"
# pattern we noted in the evaluation notebook.

def softmax_probs(logits, T=1.0):
    """Numerically-stable softmax of logits / T."""
    scaled = logits / T
    scaled = scaled - scaled.max(axis=1, keepdims=True)
    exp = np.exp(scaled)
    return exp / exp.sum(axis=1, keepdims=True)

probs_test_T1  = softmax_probs(test_logits, T=1.0)
probs_calib_T1 = softmax_probs(calib_logits, T=1.0)
scores_test_T1 = probs_test_T1[:, 1]

ece_test_T1, _  = expected_calibration_error(scores_test_T1, y_test)
ece_calib_T1, _ = expected_calibration_error(probs_calib_T1[:, 1], y_calib)
print(f"\nPRE-SCALING (T=1):")
print(f"  Test ECE:        {ece_test_T1:.4f}")
print(f"  Calibration ECE: {ece_calib_T1:.4f}")
print(f"  Test predictions in [0.01, 0.99]: "
      f"{((scores_test_T1 > 0.01) & (scores_test_T1 < 0.99)).sum()} / {len(scores_test_T1)}")

## Fit temperature on the calibration set

In [ ]:
# Optimize T by minimizing NLL of softmax(logits/T) against true labels.
# Standard L-BFGS optimization, ~50 iterations is enough.

print("\nFitting temperature on calibration set...")

calib_logits_t = torch.tensor(calib_logits, dtype=torch.float, device=device)
calib_labels_t = torch.tensor(y_calib, dtype=torch.long, device=device)

# Learnable scalar. Initialize at 1.0 (no scaling).
T = nn.Parameter(torch.ones(1, device=device))

optimizer = torch.optim.LBFGS([T], lr=0.01, max_iter=100)
criterion = nn.CrossEntropyLoss()

def closure():
    optimizer.zero_grad()
    loss = criterion(calib_logits_t / T, calib_labels_t)
    loss.backward()
    return loss

optimizer.step(closure)
T_fit = T.detach().cpu().numpy().item()
print(f"  Fitted T = {T_fit:.4f}")
print(f"    T > 1 means model was overconfident — scaling softens predictions")
print(f"    T < 1 means model was underconfident — scaling sharpens predictions")

## Apply scaling, recompute diagnostics

In [ ]:
probs_test_Tfit  = softmax_probs(test_logits, T=T_fit)
probs_calib_Tfit = softmax_probs(calib_logits, T=T_fit)
scores_test_Tfit = probs_test_Tfit[:, 1]
preds_test_Tfit  = (scores_test_Tfit >= 0.5).astype(int)

ece_test_Tfit, _  = expected_calibration_error(scores_test_Tfit, y_test)
ece_calib_Tfit, _ = expected_calibration_error(probs_calib_Tfit[:, 1], y_calib)
print(f"\nPOST-SCALING (T={T_fit:.4f}):")
print(f"  Test ECE:        {ece_test_Tfit:.4f}  (was {ece_test_T1:.4f}, "
      f"{(1 - ece_test_Tfit/ece_test_T1)*100:.1f}% reduction)")
print(f"  Calibration ECE: {ece_calib_Tfit:.4f} (was {ece_calib_T1:.4f}, "
      f"{(1 - ece_calib_Tfit/ece_calib_T1)*100:.1f}% reduction)")
print(f"  Test predictions in [0.01, 0.99]: "
      f"{((scores_test_Tfit > 0.01) & (scores_test_Tfit < 0.99)).sum()} / {len(scores_test_Tfit)}")

## Verify threshold-independent metrics are preserved

In [ ]:
# Temperature scaling is monotonic in scores, so AP and ROC-AUC should be
# IDENTICAL pre and post. F1 at threshold=0.5 may shift slightly because the
# threshold is now applied to a smoothed probability distribution.

preds_test_T1 = (scores_test_T1 >= 0.5).astype(int)
print("\nMetric preservation check (pre vs post scaling):")
print(f"  AP:        {average_precision_score(y_test, scores_test_T1):.4f}  ->  "
      f"{average_precision_score(y_test, scores_test_Tfit):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, scores_test_T1):.4f}  ->  "
      f"{roc_auc_score(y_test, scores_test_Tfit):.4f}")
print(f"  F1@0.5:    {f1_score(y_test, preds_test_T1, pos_label=1):.4f}  ->  "
      f"{f1_score(y_test, preds_test_Tfit, pos_label=1):.4f}")
print(f"  (AP and ROC-AUC SHOULD be identical — they're threshold-free.")
print(f"   F1 may shift slightly because we're applying threshold 0.5 to softened probs.)")

## Figure — pre vs post calibration diagrams

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
n_bins = 15

for ax, (label, scores, ece) in zip(
    axes,
    [(f"Before (T=1)\nECE = {ece_test_T1:.3f}", scores_test_T1, ece_test_T1),
     (f"After (T={T_fit:.3f})\nECE = {ece_test_Tfit:.3f}", scores_test_Tfit, ece_test_Tfit)],
):
    title, scores_arr, _ = label, scores, ece
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_acc = []; bin_count = []
    for i in range(n_bins):
        if i == n_bins - 1:
            mask = (scores_arr >= bin_edges[i]) & (scores_arr <= bin_edges[i+1])
        else:
            mask = (scores_arr >= bin_edges[i]) & (scores_arr < bin_edges[i+1])
        if mask.sum() > 0:
            bin_acc.append(y_test[mask].mean()); bin_count.append(mask.sum())
        else:
            bin_acc.append(np.nan); bin_count.append(0)
    bin_acc = np.array(bin_acc); bin_count = np.array(bin_count)
    sizes = 20 + 600 * (bin_count / max(bin_count.max(), 1))
    ax.plot([0, 1], [0, 1], "k--", lw=1.2, alpha=0.5, label="perfect calibration")
    valid = ~np.isnan(bin_acc)
    ax.scatter(bin_centers[valid], bin_acc[valid], s=sizes[valid],
               color="#5BB17A", alpha=0.7, edgecolor="black", linewidth=1.2)
    ax.plot(bin_centers[valid], bin_acc[valid], color="#5BB17A", lw=1.8, alpha=0.7)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Predicted P(refuse)")
    ax.set_ylabel("Empirical fraction of refuse")
    ax.set_title(label, fontsize=11)
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_aspect("equal")

fig.suptitle("DistilBERT Calibration: Before vs After Temperature Scaling\n"
             f"Fitted T = {T_fit:.3f}  →  "
             f"ECE reduced by {(1 - ece_test_Tfit/ece_test_T1)*100:.0f}%",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "09_tempscale_calibration.png", bbox_inches="tight")
plt.show()
print("Saved 09_tempscale_calibration.png")

## Figure — pre vs post score distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
bins = np.linspace(0, 1, 41)

for ax, scores, title in zip(
    axes,
    [scores_test_T1, scores_test_Tfit],
    [f"Before (T=1)", f"After (T={T_fit:.3f})"],
):
    neg = scores[y_test == 0]; pos = scores[y_test == 1]
    ax.hist(neg, bins=bins, alpha=0.6, color="#4A90D9",
            label=f"don't refuse (n={len(neg)})", edgecolor="black", linewidth=0.3)
    ax.hist(pos, bins=bins, alpha=0.6, color="#D9534F",
            label=f"refuse (n={len(pos)})", edgecolor="black", linewidth=0.3)
    ax.axvline(0.5, color="black", ls="--", lw=1, alpha=0.6, label="threshold=0.5")
    ax.set_xlabel("Predicted P(refuse)")
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8, loc="upper center")
    ax.set_yscale("log")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Count (log scale)")
fig.suptitle("DistilBERT Score Distribution: Before vs After Temperature Scaling\n"
             "Scaling spreads probability mass across the [0, 1] range",
             fontsize=12, y=1.05)
plt.tight_layout()
plt.savefig(FIG_DIR / "10_tempscale_distribution.png", bbox_inches="tight")
plt.show()
print("Saved 10_tempscale_distribution.png")

## Figure — recall vs FPR comparison

In [ ]:
# Temperature scaling preserves the ROC curve (because it's monotonic), so
# this figure should show the curves perfectly overlapping. The interesting
# difference is where the canonical threshold=0.5 lands — that's the only
# point that moves.

fig, ax = plt.subplots(figsize=(9, 5.5))

fpr1, tpr1, _ = roc_curve(y_test, scores_test_T1)
fpr2, tpr2, _ = roc_curve(y_test, scores_test_Tfit)
ax.plot(fpr1, tpr1, lw=3, alpha=0.5, label="Before scaling (T=1)", color="#999999")
ax.plot(fpr2, tpr2, lw=1.5, label=f"After scaling (T={T_fit:.3f})", color="#5BB17A", linestyle="--")

# Mark where threshold=0.5 lands on each curve
def point_at_threshold(y_true, y_score, target_thr=0.5):
    pred = (y_score >= target_thr).astype(int)
    tn = ((pred == 0) & (y_true == 0)).sum()
    fp = ((pred == 1) & (y_true == 0)).sum()
    tp = ((pred == 1) & (y_true == 1)).sum()
    fn = ((pred == 0) & (y_true == 1)).sum()
    return fp / (fp + tn), tp / (tp + fn)

fpr_T1, tpr_T1 = point_at_threshold(y_test, scores_test_T1)
fpr_Tfit, tpr_Tfit = point_at_threshold(y_test, scores_test_Tfit)
ax.scatter([fpr_T1], [tpr_T1], s=140, color="#666666", edgecolor="black", linewidth=1.5,
           zorder=5, label=f"thr=0.5 before  (FPR={fpr_T1:.3f}, rec={tpr_T1:.3f})")
ax.scatter([fpr_Tfit], [tpr_Tfit], s=140, color="#5BB17A", edgecolor="black", linewidth=1.5,
           zorder=5, label=f"thr=0.5 after  (FPR={fpr_Tfit:.3f}, rec={tpr_Tfit:.3f})")

ax.axvspan(0, 0.01, alpha=0.12, color="green")
ax.axvspan(0.01, 0.05, alpha=0.08, color="goldenrod")
ax.set_xlim(0, 0.2); ax.set_ylim(0.85, 1.005)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("Recall")
ax.set_title("DistilBERT ROC Curve: Temperature Scaling Preserves Ordering\n"
             "Curves overlap because T-scaling is monotonic. Threshold=0.5 moves to a different operating point.",
             fontsize=11)
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "11_tempscale_roc.png", bbox_inches="tight")
plt.show()
print("Saved 11_tempscale_roc.png")

## Per-bucket recall comparison (T=1 vs T=fitted)

In [ ]:
print("\nPer-bucket comparison at threshold = 0.5:")
print(f"{'Bucket':<25} {'Pre-scale':>12} {'Post-scale':>12} {'Δ':>8}")
print("-" * 60)
for bucket in sorted(test_df_clean[test_df_clean["label"] == 0]["source"].unique()):
    mask = (test_df_clean["source"] == bucket).to_numpy()
    pre_recall  = (preds_test_T1[mask]   == 0).mean()
    post_recall = (preds_test_Tfit[mask] == 0).mean()
    print(f"{bucket:<25} {pre_recall:>12.4f} {post_recall:>12.4f} "
          f"{post_recall - pre_recall:>+8.4f}")

print("\nPositive recall:")
mask_pos = (test_df_clean["label"] == 1).to_numpy()
pre_pos_recall  = (preds_test_T1[mask_pos]   == 1).mean()
post_pos_recall = (preds_test_Tfit[mask_pos] == 1).mean()
print(f"{'all positives':<25} {pre_pos_recall:>12.4f} {post_pos_recall:>12.4f} "
      f"{post_pos_recall - pre_pos_recall:>+8.4f}")

## Save scaled predictions alongside the original

In [ ]:
# These get added to the existing all_predictions_clean.parquet for any
# further analysis or for inclusion in the write-up tables.

preds_existing = pd.read_parquet(OUT_DIR / "all_predictions_clean.parquet")
# This notebook retrained DistilBERT, so its scores will differ slightly from
# the originals saved by the cleanup notebook (which trained on the full train
# set). We save them as a separate column for the write-up's calibration story
# without overwriting the comparison results.
if len(preds_existing) == len(test_df_clean):
    preds_existing["model_3_score_T1"]    = scores_test_T1
    preds_existing["model_3_score_Tfit"]  = scores_test_Tfit
    preds_existing["model_3_pred_Tfit"]   = preds_test_Tfit
    preds_existing.to_parquet(OUT_DIR / "all_predictions_clean.parquet", index=False)
    print(f"\nAppended T=1 and T=fitted scores to all_predictions_clean.parquet")
else:
    # Length mismatch shouldn't happen if cleanup was deterministic, but
    # save separately if it does.
    pd.DataFrame({
        "text": test_df_clean["text"], "label": y_test,
        "source": test_df_clean["source"],
        "score_T1":   scores_test_T1, "score_Tfit": scores_test_Tfit,
        "pred_Tfit":  preds_test_Tfit,
    }).to_parquet(OUT_DIR / "tempscale_predictions.parquet", index=False)
    print(f"\nLength mismatch — saved to tempscale_predictions.parquet instead")

## Summary

In [ ]:
print("\n" + "=" * 70)
print("  TEMPERATURE SCALING SUMMARY")
print("=" * 70)
print(f"\nFitted temperature T = {T_fit:.4f}")
print(f"\nExpected Calibration Error (ECE) on test set:")
print(f"  Before (T=1):           {ece_test_T1:.4f}")
print(f"  After  (T={T_fit:.3f}):  {ece_test_Tfit:.4f}")
print(f"  Reduction: {(1 - ece_test_Tfit/ece_test_T1)*100:.1f}%")
print(f"\nPredictions in [0.01, 0.99] (i.e. NOT pinned to extremes):")
n_middle_T1   = ((scores_test_T1 > 0.01) & (scores_test_T1 < 0.99)).sum()
n_middle_Tfit = ((scores_test_Tfit > 0.01) & (scores_test_Tfit < 0.99)).sum()
print(f"  Before: {n_middle_T1:4d} / {len(scores_test_T1)} "
      f"({100*n_middle_T1/len(scores_test_T1):.1f}%)")
print(f"  After:  {n_middle_Tfit:4d} / {len(scores_test_Tfit)} "
      f"({100*n_middle_Tfit/len(scores_test_Tfit):.1f}%)")
print(f"\nThreshold-free metrics are preserved (monotonic transformation):")
print(f"  AP:      {average_precision_score(y_test, scores_test_T1):.4f}")
print(f"  ROC-AUC: {roc_auc_score(y_test, scores_test_T1):.4f}")
print(f"\nFigures saved to {FIG_DIR}/")
for f in sorted(FIG_DIR.glob("*tempscale*.png")):
    print(f"  - {f.name}")